In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import src

/Users/gperaza/Research/informal-jobs-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Both surveys are loaded through the project's data packages instead of local CSV files:

1. **ENOE** via [`mxcensus`](https://github.com/CentroFuturoCiudades/mxcensus): `load_enoe_persons` returns the SDEM roster joined with the two occupation questionnaires (COE1, COE2) for each quarter in `src.ENOE_PERIODS` (2022 T1 – 2023 T4, pooled; the reference quarter `src.ENOE_PERIOD` = 2023 T1 matches the OD fieldwork), restricted to Jalisco (`ent == 14`). Pooling eight quarters gives ~50k workers instead of ~7k; survey weights are divided by the number of quarters so weighted totals remain the average quarterly population, and the panel structure (a dwelling is visited up to five times) is handled downstream by grouping on the cross-quarter household key (`src.ENOE_GROUP_KEYS`). `load_enoe(table="sdem")` provides the full dwelling roster used to count household size.
2. **Origin–Destination survey** via [`eodgdl`](https://github.com/CentroFuturoCiudades/eodgdl): `load_eod` returns dwellings (`viv`), persons (`hab`), trips and legs with snake_case column names; persons are joined with the dwelling attributes (municipality, AGEB, centrality, household size).

On first use each package downloads and caches the raw tables (`~/Library/Caches/mxcensus`, `~/Library/Caches/eodgdl`); set `MXCENSUS_CACHE_DIR` / `EODGDL_CACHE_DIR` to relocate the caches, or `EODGDL_DATA_DIR` to point `eodgdl` at local IMEPLAN files.

The employment definition for ENOE is a parameter of `src.generate_enoe_dataframe`:

- `"clase2"` (default): INEGI's employed population (`clase2 == 1`, i.e. worked last week or had a job and was temporarily absent) on the analytical universe: definitive interview (`r_def == 0`), habitual or new residents (`c_res in {1, 3}`) and ages 12–98 (`src.ENOE_MIN_AGE`, `src.ENOE_MAX_AGE`). INEGI publishes employment for ages 15+; the floor is lowered to 12 because the OD survey records working 12–14 year olds.
- `"p1"`: worked at least one hour last week (COE1 `p1 == 1`), no residency or age restriction — the definition used by the original pipeline, a strict subset of the default.

In [2]:
ENOE_EMPLOYMENT_FILTER = "clase2"  # "clase2" or "p1", see above

ENOE_PERIODS = src.ENOE_PERIODS  # quarters pooled for training; weights are divided by their number so totals stay at population scale

enoe = src.generate_enoe_dataframe(periods=ENOE_PERIODS, state_code=src.ENOE_STATE_CODE, employment_filter=ENOE_EMPLOYMENT_FILTER)
print("ENOE workers per quarter:", enoe["period"].value_counts().sort_index().to_dict())
od = src.generate_od_dataframe()

ENOE workers per quarter: {'2022t1': 6456, '2022t2': 6436, '2022t3': 6286, '2022t4': 6256, '2023t1': 6973, '2023t2': 6503, '2023t3': 6459, '2023t4': 6338}


For the ENOE, employed persons in Jalisco are selected, household size is computed from the full SDEM roster of each dwelling, and the employment variables and weighting factors needed later are retained (`src.ENOE_OUTPUT_COLUMNS`). All columns are integer codes.

For the OD, the persons table is joined with the dwelling table (municipality, AGEB, centrality and household size) and filtered to employed persons. Raw OD columns whose names collide with the harmonized attributes created in the next stage carry a `_raw` suffix (`src.OD_RAW_COLUMN_RENAMES`).

In [3]:
print(f"ENOE workers: {enoe.shape}")
print(f"OD workers: {od.shape}")

assert enoe["ent"].eq(src.ENOE_STATE_CODE).all(), "ENOE contains observations outside Jalisco."
assert enoe["survey_weight"].notna().all(), "ENOE contains missing survey weights."
assert od["expansion_factor"].notna().all(), "OD contains missing expansion factors."

display(enoe.head())
display(od.head())

ENOE workers: (51707, 24)
OD workers: (26913, 60)


,period,tipo,mes_cal,cd_a,ent,con,v_sel,n_hog,h_mud,n_ren,...,estrato_socioeconomico,sex,pos_ocu,scian,eda,cs_p13_1,emp_ppal,e_con,par_c,dwelling_size
0,2022t1,1,96,2,14,40001,1,1,0,1,...,40,1,1,15,49,8,2,5,101,6
1,2022t1,1,96,2,14,40001,1,1,0,2,...,40,2,1,15,28,7,2,5,201,6
2,2022t1,1,96,2,14,40001,1,1,0,4,...,40,1,1,18,25,7,1,6,413,6
3,2022t1,1,96,2,14,40001,1,1,0,6,...,40,2,1,7,50,7,2,6,403,6
4,2022t1,1,96,2,14,40001,2,1,1,1,...,40,1,2,12,59,7,2,5,101,4


,folio_vivienda,folio_habitante,fecha,salio_casa_ayer,razon_no_viaje,viajes_contados,dia_semana_viajes,parentesco_raw,sexo_nacimiento,genero_identidad,...,weekend_dest_salud,weekend_dest_banco_pagos,weekend_dest_guarderia,weekend_dest_regresar_casa,weekend_dest_religion,expansion_factor,municipio_raw,ageb,centralidad,dwelling_size
0,1,1,2023-01-25 00:00:00+00:00,Sí,<NA>,7,Jueves,Jefe del hogar,Hombres,Hombres,...,Sí,Sí,Sí,Sí,No,51,Tlajomulco,1409700251418,30F,5
1,2,1,2023-01-26 00:00:00+00:00,Sí,<NA>,2,Miércoles,Jefe del hogar,Hombres,Hombres,...,No,No,No,No,No,249,Guadalajara,1403900010859,09,3
2,2,2,2023-01-26 00:00:00+00:00,No,Otros (especifique),0,<NA>,Otro parentesco,Hombres,Hombres,...,No,No,No,No,No,249,Guadalajara,1403900010859,09,3
3,2,3,2023-01-26 00:00:00+00:00,No,Otros (especifique),0,<NA>,Compañero,Hombres,Hombres,...,No,No,No,No,No,249,Guadalajara,1403900010859,09,3
4,3,1,2023-01-26 00:00:00+00:00,No,Otros (especifique),0,<NA>,Cónyuge,Mujeres,Mujeres,...,No,No,No,No,No,249,Guadalajara,1403900010539,09,3


If a reference run is available in `outputs_baseline/`, compare the row identities and weighted totals of the new base dataframes against it (see `src/compare_outputs.py`).

In [4]:
baseline_directory = ROOT / "outputs_baseline"
output_directory = ROOT / "outputs"
output_directory.mkdir(exist_ok=True)

enoe.to_parquet(output_directory / "enoe_workers.parquet", index=False)
od.to_parquet(output_directory / "od_workers.parquet", index=False)

if baseline_directory.exists():
    src.print_comparison(src.compare_all(baseline_directory, output_directory, stage="1"))

== row_counts
        file  rows_base  rows_new  cols_base  cols_new  n_cols_only_base  n_cols_only_new cols_only_base                    cols_only_new
enoe_workers       6793     51707         22        24                 0                2             [] [estrato_socioeconomico, period]
  od_workers      26913     26913         60        60                 0                0             []                               []

== weighted_totals
        file           weight  total_base      total_new  ratio
enoe_workers    survey_weight     4028446 4,048,307.6250 1.0049
  od_workers expansion_factor     2272378 2,272,378.0000 1.0000

== key_overlap
        file  in_both  only_base  only_new
enoe_workers     6793          0     33922
  od_workers    26913          0         0

